# Step 5: Metrics & Outputs

This notebook is the **analysis layer** — it takes the raw simulation output from `04_simulation_runner.ipynb` and converts it into structured summaries and charts that answer the clinical and operational questions the model was built to address.

## What this notebook covers

**Quantitative summaries**
- Full text summary of all simulation metrics: volumes, rates, LTFU breakdown, Lung funnel.
- Key rate table: cervical screening rate, abnormal rate, colposcopy completion, treatment rate, LTFU rate.
- Age-stratified cervical result breakdown: how do result distributions differ between young (21–29) and middle (30–65) women?

**Visualizations (6 charts)**
1. Cervical screening pipeline funnel — where do patients drop off between provider visit and excisional treatment?
2. Result distribution by stratum — do simulated rates match the configured expected rates?
3. CIN grade distribution by trigger — how does the colposcopy procedure mix change with upstream result severity?
4. LTFU breakdown — where are patients most commonly lost? Three-panel: by node, outcome pie, and key rates.
5. Lung LDCT pathway funnel — step-by-step attrition from eligible patients to treated malignancies.
6. Revenue analysis — realized vs. foregone revenue, showing the financial cost of care gaps.

**Scenario comparison**
- Side-by-side fragmented vs. coordinated care rates. Currently identical (coordinated LTFU probs not yet set) — use this as the template once clinical team provides coordinated-care assumptions.

---
Run `04_simulation_runner.ipynb` first, or call `run_simulation()` in the cell below to start fresh.

In [ ]:
import sys, random
sys.path.insert(0, '../src')

import config as cfg
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# Alias our print_summary now, before %run (cell below) overwrites the name
# with Sophia's arrivals version. Using the aliased name throughout this notebook
# ensures we always call the screening metrics summary, not the arrivals summary.
from metrics import (
    initialize_metrics,
    compute_rates,
    compute_revenue,
    print_revenue_summary,
)
from metrics import print_summary       as print_screening_summary
from metrics import print_patient_trace

print('Imports OK')

## Run Simulation (or load from runner)

The cell below uses `%run` to execute `04_simulation_runner.ipynb` in its entirety and then calls `run_simulation()` for a 3-year run. This is the standard way to use this notebook independently.

**If notebook 04 is already open and has been run in the same Jupyter kernel**, you can skip this cell — `state` and `metrics` will already be defined and you can call `run_simulation()` directly. Comment out the `%run` line if you want to reuse the existing in-memory state.

The 3-year run (`sim_days=365*3`) produces enough patients for stable rate estimates. Use fewer days for quick iteration during development.

In [2]:
# Option A: run everything fresh
%run "04_simulation_runner.ipynb"
state, metrics = run_simulation(sim_days=365 * 3, seed=42)   # 3-year run
print(f"Simulation complete. Patients in state: {state['patients_created']:,}")

Core imports OK

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ARRIVAL / ACCESS SUMMARY
Total patients created:               3000

Created by type:
  outpatient:                         2082
  drop_in:                            918

Created by destination:
  pcp:                                1088
  gynecologist:                       692
  specialist:                         629
  er:                                 591

Seen by destination:
  pcp:                                600
  gynecologist:                       450
  specialist:                         300
  er:                                 375

Drop-ins converted to outpatients:    694
Critical ER returned next day:        1035
Noncritical ER scheduled outpatient:  792

Outpatient showups:                   8040
Outpatient no-shows:                  0

DAY 0
--------------------------------------------------
--- DAY 0 START ---
Patient 1 arrives as outpati

KeyError: 'patients_created'

KeyError: 'patients_created'

## Full Summary Report

`print_screening_summary(metrics)` prints the complete simulation results in one formatted block. It covers every metric category in the model: patient volumes, per-cancer screening counts, cervical result distribution (overall and by age stratum), colposcopy counts, CIN grade breakdown, treatment types, outcome totals, LTFU breakdown by node, and the full Lung LDCT pathway funnel.

This is the canonical text output of the model — the equivalent of a simulation run report. The visualisations below are derived from the same `metrics` dict and add charts on top of the numbers printed here.

In [ ]:
# Use the aliased import so Sophia's print_summary (loaded by %run above) doesn't
# interfere. print_screening_summary calls metrics.print_summary, which expects
# our metrics dict (not Sophia's state dict).
print_screening_summary(metrics)

## Key Rate Table

`compute_rates(metrics)` derives seven key performance indicators from the raw counts. These are the primary numbers a clinical or operations team would use to evaluate the screening program:

| Rate | What it measures | Good direction |
|---|---|---|
| Cervical screening rate | % of seen patients who received a cervical screen | Higher |
| Unscreened rate | % who were eligible but didn't get screened | Lower |
| Abnormal rate | % of cervical screens with a follow-up-triggering result | Calibration target |
| Colposcopy completion | % of abnormals who completed colposcopy | Higher |
| Excisional treatment rate | % of colposcopy patients who received LEEP/cone | Calibration target |
| Overall LTFU rate | % of all patients who were lost to follow-up at any node | Lower |

The "calibration target" rates should match NYP EHR-observed rates once the probability tables in `config.py` are replaced with real data.

In [4]:
rates = compute_rates(metrics)

print('Key simulation rates:')
print('-' * 55)
labels = {
    'screening_rate_cervical_pct':  'Cervical screening rate',
    'unscreened_pct':               'Unscreened rate',
    'reschedule_rate_pct':          'Reschedule rate (of unscreened)',
    'abnormal_rate_cervical_pct':   'Cervical abnormal rate',
    'colposcopy_completion_pct':    'Colposcopy completion (of abnormals)',
    'treatment_completion_pct':     'Treatment completion (of colposcopies)',
    'ltfu_rate_pct':                'Overall LTFU rate',
}
for key, label in labels.items():
    print(f'  {label:<40} {rates[key]:6.1f}%')

Key simulation rates:
-------------------------------------------------------
  Cervical screening rate                    79.8%
  Unscreened rate                             0.4%
  Reschedule rate (of unscreened)            50.5%
  Cervical abnormal rate                     12.2%
  Colposcopy completion (of abnormals)       66.1%
  Treatment completion (of colposcopies)    133.6%
  Overall LTFU rate                           7.4%


## Cervical Results by Age Stratum

USPSTF cervical screening guidelines differ by age group, which produces different result distributions between young (21–29) and middle-aged (30–65) women. Breaking results down by stratum serves two purposes:

1. **Validation** — the simulation should produce age-specific rates that match the age-stratified probability tables in `config.CERVICAL_RESULT_PROBS`. Any large divergence from the configured rates indicates a logic error.
2. **Planning** — understanding how the result mix differs by age group helps estimate colposcopy demand from each cohort separately, which is useful for capacity planning.

Note: HPV-alone results (`HPV_NEGATIVE`, `HPV_POSITIVE`) will appear in the middle stratum only, because HPV-alone testing is not offered to patients under 30.

In [ ]:
# Iterate over the actual result keys stored during the run, not a hardcoded list.
# This avoids referencing stale category names (e.g. HPV_POS_NORMAL_CYTO)
# that no longer exist in the result draw logic.
for stratum in ('young', 'middle', 'older'):
    sub = metrics['cervical_by_age_stratum'].get(stratum, {})
    if not sub:
        continue
    total = sum(sub.values())
    print(f'\nStratum: {stratum}  (n={total:,})')
    # Sort by count descending so the most common results appear first
    for result, cnt in sorted(sub.items(), key=lambda x: -x[1]):
        print(f'  {result:<30} {cnt:>6,}  ({100*cnt/max(total,1):.1f}%)')

## LTFU Funnel (Text View)

The text funnel below shows the cumulative drop-off across the cervical pathway in a linear format. Each row shows the absolute patient count at that step and the percentage drop from the previous step.

This table is the text companion to Chart 4 (LTFU visualization) and Chart 1 (pipeline funnel). Reading it top-to-bottom makes the cascade of LTFU visible: a 20% drop at "Abnormal result" means 20% of screened patients had a normal result (expected); an 80% drop at "Abnormal results" would indicate a configuration error.

The drop from "Abnormal result" → "Colposcopy completed" is the post-abnormal LTFU rate — this is the most operationally important gap because it represents patients with a confirmed abnormal finding who never received follow-up care.

In [6]:
n = max(metrics['n_patients'], 1)

funnel = [
    ('Patients seen by provider',       metrics['n_patients']),
    ('Eligible for ≥1 screening',       metrics['n_eligible_any']),
    ('Cervical screenings completed',   metrics['n_screened']['cervical']),
    ('Abnormal cervical results',       sum(
        v for k, v in metrics['cervical_results'].items() if k != 'NORMAL'
    )),
    ('Colposcopies completed',          metrics['n_colposcopy']),
    ('Excisional treatments (LEEP/cone)', metrics['n_treatment'].get('leep', 0)
                                         + metrics['n_treatment'].get('cone_biopsy', 0)),
]

print('Pathway funnel:')
print('-' * 55)
prev = None
for label, count in funnel:
    drop = f'  (↓ {100*(1-count/prev):.0f}% drop)' if prev and prev > 0 else ''
    print(f'  {label:<42} {count:>6,}{drop}')
    prev = max(count, 1)

Pathway funnel:
-------------------------------------------------------
  Patients seen by provider                  23,490
  Eligible for ≥1 screening                  23,393  (↓ 0% drop)
  Cervical screenings completed              18,737  (↓ 20% drop)
  Abnormal cervical results                   2,282  (↓ 88% drop)
  Colposcopies completed                      1,509  (↓ 34% drop)
  Excisional treatments (LEEP/cone)             375  (↓ 75% drop)


## Workflow Scenario Comparison: Fragmented vs. Coordinated Care

The model supports two workflow modes toggled via `config.WORKFLOW_MODE`:
- **Fragmented** (current state): each specialty operates independently. Patients must navigate between PCP, GYN, and specialist referrals on their own. Higher LTFU because coordination failures happen at every handoff.
- **Coordinated** (future state): a central navigation program proactively manages referrals. Lower LTFU because a navigator ensures each patient completes the next step.

**Current limitation**: the two modes currently produce identical results because the coordinated-mode LTFU probabilities haven't been set yet. The comparison cell below is a template — once the clinical team provides coordinated-care assumptions (e.g. post-abnormal LTFU drops from 20% to 8%), update `config.LTFU_PROBS` for coordinated mode and the difference will appear here.

In [7]:
results_by_mode = {}

for mode in ('fragmented', 'coordinated'):
    cfg.WORKFLOW_MODE = mode
    # TODO: adjust LTFU_PROBS and CAPACITIES for coordinated mode once
    # clinical team provides coordinated-care assumptions
    _, m = run_simulation(sim_days=365, seed=42)
    results_by_mode[mode] = compute_rates(m)

print(f'{"Metric":<42}  {"Fragmented":>14}  {"Coordinated":>14}')
print('-' * 75)
for key, label in labels.items():
    f_val = results_by_mode['fragmented'][key]
    c_val = results_by_mode['coordinated'][key]
    print(f'  {label:<40}  {f_val:>12.1f}%  {c_val:>12.1f}%')

# Reset to default
cfg.WORKFLOW_MODE = 'fragmented'

Metric                                          Fragmented     Coordinated
---------------------------------------------------------------------------
  Cervical screening rate                           79.8%          79.8%
  Unscreened rate                                    0.4%           0.4%
  Reschedule rate (of unscreened)                   50.5%          50.5%
  Cervical abnormal rate                            12.2%          12.2%
  Colposcopy completion (of abnormals)              66.1%          66.1%
  Treatment completion (of colposcopies)           133.6%         133.6%
  Overall LTFU rate                                  7.4%           7.4%


---
## Visualizations

The charts below convert the raw metric counts into visual summaries across five areas:

1. **Cervical screening pipeline** — full pathway funnel from patients seen through to excisional treatment, showing the volume drop at each step.
2. **Cervical result distribution** — observed vs. expected result category mix, stratified by age group.
3. **CIN grade distribution by trigger** — how the colposcopy CIN grade mix shifts as the triggering result escalates in severity.
4. **LTFU by decision node** — where in the pathway are patients most commonly lost to follow-up?
5. **Revenue analysis** — realized vs. foregone procedure revenue, showing the financial impact of care gaps.

In [ ]:
## Chart 1 — Cervical Screening Pipeline (Funnel)
#
# Each bar represents one gate in the cervical pathway.
# The gap between consecutive bars is the drop-off — patients who were either
# ineligible, not due, lost to follow-up, or had a normal result that ended
# their pathway at that step. The funnel shape is expected; extremely steep
# drops highlight where care coordination investments would have the most impact.

_NORMAL_CERVICAL = {"NORMAL", "HPV_NEGATIVE"}
total_abnormal = sum(
    v for k, v in metrics['cervical_results'].items()
    if k not in _NORMAL_CERVICAL
)
cerv_excisional = (
    metrics['n_treatment'].get('leep', 0)
    + metrics['n_treatment'].get('cone_biopsy', 0)
)

# Build the funnel — each tuple is (label, count)
funnel_stages = [
    ('Patients seen',                   metrics['n_patients']),
    ('Eligible (≥1 screen)',            metrics['n_eligible_any']),
    ('Cervical screened',               metrics['n_screened']['cervical']),
    ('Abnormal result',                 total_abnormal),
    ('Colposcopy completed',            metrics['n_colposcopy']),
    ('Excisional treatment\n(LEEP/cone)', cerv_excisional),
]

labels  = [s[0] for s in funnel_stages]
counts  = [s[1] for s in funnel_stages]
n_start = max(counts[0], 1)
pcts    = [c / n_start * 100 for c in counts]

# Colour gradient: blue at the top funnelling to teal-green at the bottom
colors = ['#1565C0', '#1976D2', '#1E88E5', '#FF8F00', '#EF5350', '#66BB6A']

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(labels[::-1], pcts[::-1], color=colors[::-1])
ax.set_xlabel('% of patients seen by a provider')
ax.set_title('Cervical Screening Pipeline — Funnel View', fontsize=13, fontweight='bold')
ax.set_xlim(0, 115)

# Annotate each bar with both the % and the absolute patient count
for bar, pct, cnt in zip(bars, pcts[::-1], counts[::-1]):
    ax.text(pct + 1, bar.get_y() + bar.get_height() / 2,
            f'{pct:.1f}%  (n={cnt:,})', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
## Chart 2 — Cervical Result Distribution by Age Stratum
#
# Two side-by-side bar charts, one per age stratum (young 21–29, middle 30–65).
# Each chart shows the observed result category frequency from the simulation.
# The expected rates from config are overlaid as a reference bar so any
# calibration drift is immediately visible.
#
# If observed ≈ expected, the probability tables in config are being applied
# correctly. Systematic gaps between bars indicate a bug or a miscalibrated table.

strata_config = {
    'young':  ('young (21–29)',  'young'),
    'middle': ('middle (30–65)', 'middle_cytology'),
}
# Cytology Bethesda categories only — HPV-alone result categories are plotted separately
cyto_cats = ['NORMAL', 'ASCUS', 'LSIL', 'ASC-H', 'HSIL']
hpv_cats  = ['HPV_NEGATIVE', 'HPV_POSITIVE']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (stratum, (stratum_label, cfg_key)) in zip(axes[:2], strata_config.items()):
    sub   = metrics['cervical_by_age_stratum'].get(stratum, {})
    total = max(sum(sub.values()), 1)

    # Only show cytology categories for this stratum panel
    cats   = [c for c in cyto_cats if c in sub or cfg_key in cfg.CERVICAL_RESULT_PROBS]
    obs    = [sub.get(c, 0) / total * 100 for c in cats]
    exp    = [cfg.CERVICAL_RESULT_PROBS.get(cfg_key, {}).get(c, 0) * 100 for c in cats]

    x = range(len(cats))
    ax.bar([i - 0.2 for i in x], obs, width=0.4, label='Observed', color='#1976D2', alpha=0.85)
    ax.bar([i + 0.2 for i in x], exp, width=0.4, label='Expected', color='#90CAF9', alpha=0.85)
    ax.set_xticks(list(x))
    ax.set_xticklabels(cats, rotation=30, ha='right')
    ax.set_ylabel('% of screenings')
    ax.set_title(f'Cytology — {stratum_label}\n(n={sum(sub.values()):,})')
    ax.legend(fontsize=8)

# HPV-alone panel — uses the middle_hpv table, only HPV_NEGATIVE / HPV_POSITIVE
ax = axes[2]
middle_sub = metrics['cervical_by_age_stratum'].get('middle', {})
total_mid  = max(sum(middle_sub.values()), 1)
obs_hpv = [middle_sub.get(c, 0) / total_mid * 100 for c in hpv_cats]
exp_hpv = [cfg.CERVICAL_RESULT_PROBS['middle_hpv'].get(c, 0) * 100 for c in hpv_cats]

x = range(len(hpv_cats))
ax.bar([i - 0.2 for i in x], obs_hpv, width=0.4, label='Observed', color='#1565C0', alpha=0.85)
ax.bar([i + 0.2 for i in x], exp_hpv, width=0.4, label='Expected', color='#90CAF9', alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(hpv_cats, rotation=15, ha='right')
ax.set_ylabel('% of HPV-alone screenings')
ax.set_title('HPV-alone — middle (30–65)')
ax.legend(fontsize=8)

plt.suptitle('Cervical Result Distribution: Observed vs. Expected', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
## Chart 3 — CIN Grade Distribution by Triggering Result (Stacked Bar)
#
# Each bar represents the subset of colposcopy patients whose referral was triggered
# by a specific cervical result category. The coloured segments show what fraction
# of that group was found to have each CIN grade.
#
# This chart is used to validate the model and to communicate to clinical teams
# how the expected procedure mix (surveillance vs. LEEP) varies with the upstream
# result. A higher-grade trigger (HSIL) should show a larger CIN2/3 fraction,
# meaning more LEEP procedures per colposcopy performed.
#
# Note: the colposcopy_results dict is overall — it aggregates across all triggers.
# We reconstruct a per-trigger view from the config expected tables here.

if metrics['n_colposcopy'] > 0:
    triggers   = ['ASCUS', 'LSIL', 'ASC-H', 'HSIL', 'HPV_POSITIVE']
    cin_grades = ['NORMAL', 'CIN1', 'CIN2', 'CIN3']
    colors_cin = ['#4CAF50', '#FFC107', '#FF5722', '#B71C1C']   # green → dark red

    # Derive per-trigger CIN fractions from config (expected proportions).
    # The simulation colposcopy_results dict doesn't split by trigger, so we use
    # the config tables here as a model-implied view — these are the same tables
    # that drive the stochastic draws in followup.run_colposcopy().
    fig, ax = plt.subplots(figsize=(10, 5))
    bottoms = [0.0] * len(triggers)

    for i, grade in enumerate(cin_grades):
        vals = []
        for t in triggers:
            probs = cfg.COLPOSCOPY_RESULT_PROBS.get(f'from_{t}', {})
            vals.append(probs.get(grade, 0) * 100)
        ax.bar(triggers, vals, bottom=bottoms, color=colors_cin[i], label=grade, alpha=0.9)
        bottoms = [b + v for b, v in zip(bottoms, vals)]

    ax.set_ylabel('CIN grade distribution (%)')
    ax.set_title('Expected CIN Grade Mix by Triggering Result\n(from config probability tables)',
                 fontsize=12, fontweight='bold')
    ax.legend(title='CIN grade', loc='upper right')
    ax.set_ylim(0, 110)

    # Annotate total colposcopies from this simulation run
    ax.text(0.01, 0.97,
            f'Total colposcopies in run: {metrics["n_colposcopy"]:,}',
            transform=ax.transAxes, va='top', fontsize=9, color='#333')

    plt.tight_layout()
    plt.show()
else:
    print('No colposcopies performed in this simulation run.')

In [ ]:
## Chart 4 — Loss-to-Follow-Up Breakdown
#
# Three panels showing the LTFU problem from different angles:
#
#   Left:  LTFU count at each named node — post-abnormal (never books colposcopy),
#          post-colposcopy (never completes treatment), and declined screening entirely.
#          The biggest bar is the priority target for care coordination.
#
#   Middle: Outcome breakdown — of all patients who exited the system, what fraction
#           were treated, untreated (lost after malignancy confirmation), or LTFU?
#
#   Right:  Key rate summary — screening rate, abnormal rate, and colposcopy completion
#           rate displayed as a horizontal reference bar chart. These three rates together
#           describe the overall health of the screening program.

rates = compute_rates(metrics)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Panel 1: LTFU by node ─────────────────────────────────────────────────────
ax = axes[0]
ltfu_nodes  = ['Post-abnormal\n(no colposcopy)', 'Post-colposcopy\n(no treatment)', 'Declined\nscreening']
ltfu_counts = [metrics['ltfu_post_abnormal'], metrics['ltfu_post_colposcopy'], metrics['ltfu_unscreened']]
ltfu_colors = ['#EF5350', '#B71C1C', '#BDBDBD']
bars = ax.bar(ltfu_nodes, ltfu_counts, color=ltfu_colors)
ax.set_ylabel('Patients lost at this node')
ax.set_title('LTFU Count by Node', fontsize=11, fontweight='bold')
for bar, cnt in zip(bars, ltfu_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{cnt:,}', ha='center', fontsize=9)

# ── Panel 2: Patient outcome pie ──────────────────────────────────────────────
ax = axes[1]
outcome_labels = ['Treated\n(LEEP/surgery)', 'LTFU', 'Untreated\n(declined tx)']
outcome_counts = [metrics['n_treated'], metrics['n_ltfu'], metrics['n_untreated']]
outcome_colors = ['#66BB6A', '#EF5350', '#FF8F00']
# Only draw pie if there are any outcomes to show
nonzero = [(l, c, col) for l, c, col in zip(outcome_labels, outcome_counts, outcome_colors) if c > 0]
if nonzero:
    lbls, cnts, cols = zip(*nonzero)
    ax.pie(cnts, labels=lbls, colors=cols, autopct='%1.1f%%', startangle=90,
           textprops={'fontsize': 9})
    ax.set_title('Patient Outcomes\n(all exited patients)', fontsize=11, fontweight='bold')
else:
    ax.text(0.5, 0.5, 'No exited patients', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Patient Outcomes')

# ── Panel 3: Key rate bar chart ───────────────────────────────────────────────
ax = axes[2]
rate_labels = [
    'Cervical\nscreening rate',
    'Abnormal\nresult rate',
    'Colposcopy\ncompletion\n(of abnormals)',
    'Excisional tx\n(of colposcopies)',
]
rate_values = [
    rates['screening_rate_cervical_pct'],
    rates['abnormal_rate_cervical_pct'],
    rates['colposcopy_completion_pct'],
    rates['treatment_completion_pct'],
]
rate_colors = ['#1976D2', '#FF8F00', '#7B1FA2', '#1B5E20']
bars2 = ax.bar(rate_labels, rate_values, color=rate_colors, alpha=0.85)
ax.set_ylabel('Rate (%)')
ax.set_title('Key Performance Rates', fontsize=11, fontweight='bold')
ax.set_ylim(0, 115)
for bar, val in zip(bars2, rate_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', fontsize=9)

plt.suptitle('Loss-to-Follow-Up and Outcome Summary', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
## Chart 5 — Lung LDCT Pathway Funnel (if lung patients were seen)
#
# Shows cumulative attrition across the lung pathway: from eligible patients
# through LDCT order, scheduling, scan completion, result communication, and
# (for RADS 4 cases) through biopsy to malignancy confirmation and treatment.
#
# Each bar is annotated with the absolute count and percentage of the eligible
# starting cohort. The step-to-step drop-off is the LTFU rate at that specific node.

if metrics['lung_eligible'] > 0:
    lung_steps = [
        ('Eligible (50–80, ≥20 pk-yrs)',  'lung_eligible'),
        ('LDCT order placed',              'lung_referral_placed'),
        ('LDCT scheduled',                 'lung_ldct_scheduled'),
        ('LDCT completed',                 'lung_ldct_completed'),
        ('Result communicated',            'lung_result_communicated'),
        ('Biopsy referral (RADS 4)',       'lung_biopsy_referral'),
        ('Biopsy scheduled',               'lung_biopsy_scheduled'),
        ('Biopsy completed',               'lung_biopsy_completed'),
        ('Malignancy confirmed',           'lung_malignancy_confirmed'),
        ('Treatment given',                'lung_treatment_given'),
    ]
    # Only include steps with at least one patient (zeros after biopsy referral
    # are expected in short runs with few RADS 4 patients)
    lung_steps = [(lbl, key) for lbl, key in lung_steps if metrics[key] > 0]

    lbls   = [s[0] for s in lung_steps]
    cnts   = [metrics[s[1]] for s in lung_steps]
    n_elig = max(cnts[0], 1)

    # Build a colour gradient: blues for LDCT steps, red for malignancy, green for treatment
    palette = plt.cm.Blues_r(range(40, 220, max(1, 180 // len(cnts))))
    if len(cnts) >= 2:
        palette[-1] = (0.40, 0.73, 0.42, 1.0)   # treatment — green
    if len(cnts) >= 3 and metrics['lung_malignancy_confirmed'] > 0:
        # Find malignancy step index and colour it red
        for i, (_, key) in enumerate(lung_steps):
            if key == 'lung_malignancy_confirmed':
                palette[-(len(cnts) - i)] = (0.90, 0.33, 0.32, 1.0)
                break

    fig, ax = plt.subplots(figsize=(11, max(4, len(cnts) * 0.65)))
    bars = ax.barh(lbls[::-1], [c / n_elig * 100 for c in cnts[::-1]], color=palette[::-1])
    ax.set_xlabel('% of eligible lung patients')
    ax.set_title('Lung LDCT Pathway Funnel — Simulation Output', fontsize=12, fontweight='bold')
    ax.set_xlim(0, 120)

    for bar, cnt in zip(bars, cnts[::-1]):
        pct = cnt / n_elig * 100
        ax.text(pct + 1, bar.get_y() + bar.get_height() / 2,
                f'{cnt:,}  ({pct:.1f}%)', va='center', fontsize=9)

    plt.tight_layout()
    plt.show()
else:
    print('No lung-eligible patients were processed in this simulation run.')

In [ ]:
## Chart 6 — Revenue Analysis: Realized vs. Foregone
#
# This chart converts procedure volumes into estimated dollar amounts using
# CPT-based rates from config.PROCEDURE_REVENUE (all PLACEHOLDERS).
#
# Two grouped bars side by side per category:
#   Realized  — revenue from procedures that were actually completed.
#   Foregone  — estimated revenue lost because patients dropped out at LTFU nodes.
#
# The revenue capture rate (realized / total addressable) is printed above the chart.
# A low capture rate means a large share of potential revenue is being lost to LTFU —
# which also corresponds directly to patients who are not receiving the care they need.
# Improving care coordination closes both the clinical gap and the revenue gap.

rev = compute_revenue(metrics)

# Realized revenue by procedure category
real_items = {k: v for k, v in rev['realized_by_procedure'].items() if v > 0}
fore_items = {k: v for k, v in rev['foregone_by_node'].items()      if v > 0}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: realized by procedure ───────────────────────────────────────────────
ax = axes[0]
if real_items:
    lbls  = [k.replace('_', '\n') for k in real_items]
    amts  = list(real_items.values())
    bars  = ax.bar(lbls, [a / 1000 for a in amts], color='#1976D2', alpha=0.85)
    ax.set_ylabel('Revenue ($k)')
    ax.set_title(f'Realized Revenue by Procedure\nTotal: ${rev["realized_total"]:,.0f}',
                 fontsize=11, fontweight='bold')
    for bar, amt in zip(bars, amts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f'${amt/1000:.0f}k', ha='center', fontsize=8)
    plt.setp(ax.get_xticklabels(), fontsize=8)

# ── Right: foregone by node ───────────────────────────────────────────────────
ax = axes[1]
if fore_items:
    lbls2 = [k.replace('_', '\n') for k in fore_items]
    amts2 = list(fore_items.values())
    bars2 = ax.bar(lbls2, [a / 1000 for a in amts2], color='#EF5350', alpha=0.85)
    ax.set_ylabel('Revenue foregone ($k)')
    ax.set_title(f'Foregone Revenue by LTFU Node\nTotal: ${rev["foregone_total"]:,.0f}',
                 fontsize=11, fontweight='bold')
    for bar, amt in zip(bars2, amts2):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f'${amt/1000:.0f}k', ha='center', fontsize=8)
    plt.setp(ax.get_xticklabels(), fontsize=8)

# Compute capture rate and display as suptitle
total_addressable = rev['realized_total'] + rev['foregone_total']
if total_addressable > 0:
    capture_pct = 100 * rev['realized_total'] / total_addressable
    plt.suptitle(
        f'Revenue Analysis (PLACEHOLDER CPT rates)\n'
        f'Capture rate: {capture_pct:.1f}%  |  '
        f'Foregone: {100 - capture_pct:.1f}%',
        fontsize=12, fontweight='bold'
    )

plt.tight_layout()
plt.show()

# Also print the full text summary
print_revenue_summary(metrics)